In [ ]:
import torch
import numpy as np
import pandas as pd
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


In [ ]:
trainDataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
testDataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.48MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 133kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.47MB/s]


In [ ]:
trainDataLoader = DataLoader(trainDataset, batch_size=64, shuffle=True)
testDataLoader = DataLoader(testDataset, batch_size=64, shuffle=False)

In [ ]:
class cnnModel(nn.Module):
  def __init__(self, op):
    super().__init__()

    self.Model = nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),
        nn.Flatten(),
        nn.Linear(64 * 8 * 8, 128),
        nn.ReLU(),
        nn.Linear(128, op)
    )
  def forward(self,x):
    return self.Model(x)

In [ ]:
cnnmodel = cnnModel(len(trainDataset.classes)).to(device)

In [ ]:
criteria = nn.CrossEntropyLoss()

In [ ]:
cnnOptimizer = optim.Adam(cnnmodel.parameters(), lr=0.001)

In [ ]:
epochs = 5

In [ ]:
for epoch in range(epochs):
  totalLoss = 0.0
  size = 0
  for img, label in trainDataLoader:

    img = img.to(device)
    label = label.to(device)

    op = cnnmodel(img)

    loss = criteria(op, label)
    totalLoss += loss.item()
    size +=1

    cnnOptimizer.zero_grad()
    loss.backward()
    cnnOptimizer.step()

  batch = len(trainDataLoader)
  print(f"epoch: {epoch+1}/{epochs}, size:{size}, batch: {batch}, Loss: {(totalLoss/size):.4}")



epoch: 1/5, size:938, batch: 938, Loss: 0.01083
epoch: 2/5, size:938, batch: 938, Loss: 0.009702
epoch: 3/5, size:938, batch: 938, Loss: 0.006823
epoch: 4/5, size:938, batch: 938, Loss: 0.006851
epoch: 5/5, size:938, batch: 938, Loss: 0.006641


In [ ]:
cnnmodel.eval()

correct =0
with torch.no_grad():
  for img, label in testDataLoader:

    img = img.to(device)
    label = label.to(device)

    op = cnnmodel(img)

    _,prediction = torch.max(op,1)

    correct += (prediction == label).sum().item()

print(f"Accuracy: {(correct/len(testDataset)):.2%}")


Accuracy: 99.08%
